In [ ]:
# Colab setup: install pyedautils; elsewhere load plotly.js from a CDN for the plots
import sys
if "google.colab" in sys.modules:
    %pip install -q "pyedautils>=0.0.32"
else:
    import plotly.io as pio
    pio.renderers.default = "notebook_connected"

# Median Weeks

```{admonition} Goal
:class: tip
Show the median weekly electricity consumption pattern as a heatmap with hours on one axis and weekdays on the other.
```

```{admonition} Data Basis
:class: note
The dataset `flatElectricity.csv` contains cumulative electricity meter readings at 15-minute intervals. We convert cumulative readings to consumption values for `FlatA_Ele` and sum them per hour (complete hours only).
```

## Data Preparation

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/retomarek/edap/main/edap/sampleData/flatElectricity.csv"
raw = pd.read_csv(url, sep=";")

# Parse timestamp and set as index
raw["time"] = pd.to_datetime(raw["time"])
raw = raw.set_index("time")

# Consumption per 15 minutes from the cumulative meter readings (kWh)
consumption = raw["FlatA_Ele"].diff()

# Hourly sums of complete hours only (all 4 quarter-hours have a reading)
hourly = consumption.resample("h").sum(min_count=4).dropna()

# Build clean 2-column DataFrame [timestamp, value]
df = hourly.reset_index()
df.columns = ["timestamp", "value"]
df.head()

## Visualization

In [ ]:
from pyedautils.plots.heatmaps import plot_heatmap_median_weeks

fig = plot_heatmap_median_weeks(df, title="Electricity — Median Weeks", ylab="Electricity [kWh/h]")
fig.show()

::::{dropdown} Customization
The `plot_heatmap_median_weeks` function accepts the following optional parameters:

- `colorscale`: Change the color palette (e.g., `"Viridis"`, `"Plasma"`).
- `seasons`: Custom season titles, e.g. in another language, in the order spring, summer, fall, winter.

```python
fig = plot_heatmap_median_weeks(df, title="Electricity — Median Weeks", ylab="Electricity [kWh/h]",
                                colorscale="Plasma", seasons=["Frühling", "Sommer", "Herbst", "Winter"])
```
::::